# Imports

#### Loading the data from Kaggle 

In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import scipy
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline 
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from scipy.stats import loguniform
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
import polars as pl

#model saving
import joblib
import pickle

# Data processsing

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kundanbedmutha/exam-score-prediction-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/ender/.cache/kagglehub/datasets/kundanbedmutha/exam-score-prediction-dataset/versions/2


In [3]:
df_raw = pd.read_csv(f"{path}/Exam_Score_Prediction.csv")

In [5]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        20000 non-null  int64  
 1   age               20000 non-null  int64  
 2   gender            20000 non-null  object 
 3   course            20000 non-null  object 
 4   study_hours       20000 non-null  float64
 5   class_attendance  20000 non-null  float64
 6   internet_access   20000 non-null  object 
 7   sleep_hours       20000 non-null  float64
 8   sleep_quality     20000 non-null  object 
 9   study_method      20000 non-null  object 
 10  facility_rating   20000 non-null  object 
 11  exam_difficulty   20000 non-null  object 
 12  exam_score        20000 non-null  float64
dtypes: float64(4), int64(2), object(7)
memory usage: 2.0+ MB


In [7]:
df_raw.head(5)

,student_id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,1,17,male,diploma,2.78,92.9,yes,7.4,poor,coaching,low,hard,58.9
1,2,23,other,bca,3.37,64.8,yes,4.6,average,online videos,medium,moderate,54.8
2,3,22,male,b.sc,7.88,76.8,yes,8.5,poor,coaching,high,moderate,90.3
3,4,20,other,diploma,0.67,48.4,yes,5.8,average,online videos,low,moderate,29.7
4,5,20,female,diploma,0.89,71.6,yes,9.8,poor,coaching,low,moderate,43.7


In [8]:
df_raw.drop(labels=["course", "study_method", "facility_rating", "student_id"], axis=1, inplace=True)

df = df_raw.copy()

mapping_gender = {"male" : 0, "other" : 1, "female" : 2}
df['gender'] = df_raw['gender'].map(mapping_gender)

mapping_internet_acc = {"yes" : 1, "no" : 0}
df['internet_access'] = df_raw['internet_access'].map(mapping_internet_acc)

mapping_sleep_q = {"poor" : 0, "average" : 1, "good" : 2}
df['sleep_quality'] = df_raw['sleep_quality'].map(mapping_sleep_q)

#mapping_study_m = {"coaching" : 0, "online videos" : 1, "mixed" : 2, "self-study" : 3, "group study" : 4}
#df['study_method'] : df_raw['study_method'].map(mapping_study_m)

mapping_exam_d = {"easy" : 0, "moderate" : 1, "hard" : 2}
df['exam_difficulty'] =  df_raw['exam_difficulty'].map(mapping_exam_d)

In [9]:
df.head(5)

,age,gender,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,exam_difficulty,exam_score
0,17,0,2.78,92.9,1,7.4,0,2,58.9
1,23,1,3.37,64.8,1,4.6,1,1,54.8
2,22,0,7.88,76.8,1,8.5,0,1,90.3
3,20,1,0.67,48.4,1,5.8,1,1,29.7
4,20,2,0.89,71.6,1,9.8,0,1,43.7


#### train test split

In [10]:
target = "exam_score"
features = ["study_hours", "class_attendance", "sleep_hours", "sleep_quality"]

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y)

# Modeling